In [32]:
!pip install anthropic mcp -q


In [33]:
import anthropic
import mcp
print("anthropic ok:", anthropic)
print("mcp ok:", mcp)


anthropic ok: <module 'anthropic' from '/usr/local/lib/python3.12/dist-packages/anthropic/__init__.py'>
mcp ok: <module 'mcp' from '/usr/local/lib/python3.12/dist-packages/mcp/__init__.py'>


In [34]:
def convert_currency(amount, from_currency, to_currency):

  rates = {
      "USD": 1.0,
      "INR": 83.5,
      "EUR": 0.92,
      "GBP": 0.79,
      "JPY": 149.5,

  }

  if from_currency not in rates or to_currency not in rates:
    return "Currency not supported"

  in_usd = amount / rates[from_currency]
  result = in_usd * rates[to_currency]
  return round(result, 2)


print(convert_currency(100, "USD", "INR"))
print(convert_currency(500, "INR", "USD"))



8350.0
5.99


In [35]:
!pip install anthropic -q

In [42]:
from google.colab import userdata
import anthropic
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

tools = [
    {
        "name":"convert_currency",
        "description":  "Converts an amount from one currency to another. Supoorts USD, INR, EUR, GBP, JPY.",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Thee amount of convert"},
                "from_currency": {"type": "string", "description": "Currency to convert from e.g. USD"},
                "to_currency": {"type": "string", "description": "Currency to convert to e.g. INR"}
            },
                "required": ["amount", "from_currency", "to_currency"]

        }

    }
]

print("Tools registered!")

Tools registered!


In [50]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    tools=tools,
    messages=[
        {"role": "user", "content": "How much is 100 USD in INR?"}

    ]

)

print("Stop reason:", response.stop_reason)
print("Response:", response.content)

Stop reason: tool_use
Response: [ToolUseBlock(id='toolu_01W3zrLuUYPKWuuqpg7NCYFR', caller=DirectCaller(type='direct'), input={'amount': 100, 'from_currency': 'USD', 'to_currency': 'INR'}, name='convert_currency', type='tool_use')]


In [49]:
tool_call = response.content[0]
tool_input = tool_call.input


tool_result = convert_currency(
    tool_input["amount"],
    tool_input["from_currency"],
    tool_input["to_currency"]

)
print()




AttributeError: 'TextBlock' object has no attribute 'input'

In [48]:
tool_call = response.content[0]
tool_input = tool_call.input


tool_result = convert_currency(
    tool_input["amount"],
    tool_input["from_currency"],
    tool_input["to_currency"]

)
print("Tool result:", tool_result)


final_response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "How much is 100 USD  IN INR?"},
        {"role": "assistant", "content": response.content},
        {"role": "user", "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_call.id,
                    "content": str(tool_result)
                }
            ]
        }
    ]
)

print("Claude says:", final_response.content[0]. text)

AttributeError: 'TextBlock' object has no attribute 'input'

In [47]:
def get_weather(city):

  weather_data = {
      "Delhi": {"temp": "38 C", "condition": "Hot and sunny"},
      "Tokyo": {"temp": "22 C", "condition": "Mild and Cloudy" },
      "London": {"temp": "15 C", "condition": "Rainy"},
      "Paris": {"temp": "18 C", "condition": "Partly cloudy"},
      "Dubai": {"temp": "42 C", "condition": "Very hot"},

  }
  if city not in weather_data:
    return f"Weather data not avaiable for {city}"
  w = weather_data[city]
  return f"{city}: {w['temp']}, {w['condition']}"

def estimate_flight_cost(from_city, to_city):
  costs_usd = {
      ("Delhi", "Tokyo"): 800,
      ("Delhi", "London"): 600,
      ("Delhi", "Dubai"):  200,
      ("Delhi", "Paris"): 650,
      ("Tokyo", "London"): 900,
      ("Tokyo", "Dubai"): 700
  }
  key = (from_city, to_city)
  reverse_key = (to_city, from_city)
  if key in costs_usd:
    return f"Estimated flight cost: ${costs_usd[key]} USD"
  elif reverse_key in costs_usd:
    return f"Estimated fligth cost: ${costs_usd[costs_usd[reverse_key]]} USD"
  else:
    return "Route not avaiable"

print(get_weather("Tokyo"))
print(estimate_flight_cost("Delhi", "Tokyo"))





Tokyo: 22 C, Mild and Cloudy
Estimated flight cost: $800 USD


In [46]:
all_tools = [
    {
        "name": "convert_currency",
        "description": "Converts an amount from one currency to another.Suports USD, INR, EUR, GBP, JPY.",
        "input_schema":{
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"}

            },
            "required": ["amount", "from_currency", "to_cuurency"]

        }
    },
    {
        "name": "get_weather",
        "description": "Gets the weather forecast for a city. Supports Delhi, Tokyo, London, Paris, Dubai.",
        "input_schema": {
            "type": "object",
            "properties": {
              "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"]

            }
        },
        {
            "name": "estimate_flight_cost",
            "description": "Estimates flight cost between two cities in USD.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "from_city": {"type": "string"},
                    "to_city": {"type": "string"},

                },
                "required": ["from_city", "to_city"]

            }
        }
]
print(f"Registered{len(all_tools)} tools!")


Registered3 tools!


In [45]:
from google.colab import userdata
import anthropic, os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

# Tool dispatcher
def run_tool(name, inputs):
    if name == "convert_currency":
        return convert_currency(**inputs)
    elif name == "get_weather":
        return get_weather(**inputs)
    elif name == "estimate_flight_cost":
        return estimate_flight_cost(**inputs)

# All 3 tools
all_tools = [
    {"name": "convert_currency", "description": "Converts currency. Supports USD, INR, EUR, GBP, JPY.", "input_schema": {"type": "object", "properties": {"amount": {"type": "number"}, "from_currency": {"type": "string"}, "to_currency": {"type": "string"}}, "required": ["amount", "from_currency", "to_currency"]}},
    {"name": "get_weather", "description": "Gets weather for a city. Supports Delhi, Tokyo, London, Paris, Dubai.", "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}},
    {"name": "estimate_flight_cost", "description": "Estimates flight cost between two cities.", "input_schema": {"type": "object", "properties": {"from_city": {"type": "string"}, "to_city": {"type": "string"}}, "required": ["from_city", "to_city"]}}
]

# Ask Claude
messages = [{"role": "user", "content": "I'm travelling from Delhi to Tokyo. What's the weather there, how much is the flight, and convert that flight cost to INR?"}]

response = client.messages.create(model="claude-haiku-4-5", max_tokens=1024, tools=all_tools, messages=messages)

while response.stop_reason == "tool_use":
    messages.append({"role": "assistant", "content": response.content})
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            result = run_tool(block.name, block.input)
            print(f"🔧 Called {block.name} → {result}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
    messages.append({"role": "user", "content": tool_results})
    response = client.messages.create(model="claude-haiku-4-5", max_tokens=1024, tools=all_tools, messages=messages)

print("\n🤖 Claude says:")
print(response.content[0].text)

🔧 Called get_weather → Tokyo: 22 C, Mild and Cloudy
🔧 Called estimate_flight_cost → Estimated flight cost: $800 USD
🔧 Called convert_currency → 66800.0

🤖 Claude says:
Perfect! Here's the information for your trip from Delhi to Tokyo:

**Weather in Tokyo:**
- Temperature: 22°C
- Conditions: Mild and Cloudy

**Flight Cost:**
- USD: $800
- INR: ₹66,800

The weather looks pleasant for your trip! Pack light layers since it will be mild and cloudy.


In [44]:
from google.colab import userdata
import anthropic, os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

# Simple test
r = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=10,
    messages=[{"role": "user", "content": "say hi"}]
)
print(r.content[0].text)

Hi! 👋 How can I help
